# Concept quality: crop vs. circle (VOC, COCO)

In [ ]:
import os
import numpy as np
import pandas as pd
from eval_utils import (PATHS, METHODS, N_SUBSET, VOC_CLASSES, COCO_CLASSES, CLASS_NAMES, vocab_idx,
                        VOC_SYNONYMS, COCO_SYNONYMS, SYNONYMS, check_synonyms,
                        draw_subset, load_subset, aggregate, evaluate)

print({ds: {k: os.path.exists(p) for k, p in v.items()} for ds, v in PATHS.items()})
check_synonyms(VOC_SYNONYMS, VOC_CLASSES, "VOC")
check_synonyms(COCO_SYNONYMS, COCO_CLASSES, "COCO")

In [ ]:
# ── Table (dataset × method × aggregation) + summary ──

all_rows = []
for ds in ("voc", "coco"):
    rows, labels, n_split, n_eligible = draw_subset(ds)
    data = load_subset(ds, rows)
    assert np.array_equal(data["crop_num_obj"], data["circle_num_obj"]), "Crop/Circle: different detections"
    min_w = min(data["full"].min(), *(o.min() for m in METHODS for o in data[m] if len(o)))
    assert min_w >= 0, "negative SpLiCE weights – check definition of count/active"

    res, removed = evaluate(data, labels, ds)
    print(f"{ds.upper()}: test split {n_split} | with GT label {n_eligible} | subset {N_SUBSET} "
          f"| mean objects {data['crop_num_obj'].mean():.2f} | no detection {(data['crop_num_obj'] == 0).sum()}")
    print(f"      non-coverable labels removed: {removed['labels_removed']} "
          f"(in {removed['images_affected']} images) | images excluded: {removed['images_excluded']} "
          f"| evaluated (image, label) pairs: {removed['pairs']}")
    for r in res:
        r["Dataset"] = ds.upper()
    all_rows += res

results = pd.DataFrame(all_rows).set_index(["Dataset", "Method", "Aggregation"])
display(results.round(3))

print("\nSummary (circle − crop, same subset):")
for ds in results.index.get_level_values(0).unique():
    for agg in ("Sum", "Count"):
        crop, circle = results.loc[(ds, "Crop", agg)], results.loc[(ds, "Circle", agg)]
        parts = [f"{m} {circle[m] - crop[m]:+.3f}" for m in ("Hit@10", "Hit@20", "Found", "Recall@10", "Recall@20")]
        parts.append(f"MeanRank {circle['MeanRank'] - crop['MeanRank']:+.1f} (lower = better)")
        parts.append(f"active concepts {circle['ActiveConcepts'] - crop['ActiveConcepts']:+.1f}")
        better = "Circle" if circle["Recall@10"] > crop["Recall@10"] else "Crop"
        print(f"  {ds:<5}{agg:<6}: {', '.join(parts)}  → better at Recall@10: {better}")

In [ ]:
# ── Cosine similarity: object concept vector vs. full-image concept vector ──
def cosine_to_full(full, objs):
    obj_norm = np.linalg.norm(objs, axis=1)
    valid = obj_norm > 0                       
    sims = objs[valid] @ full / (obj_norm[valid] * np.linalg.norm(full))
    return sims, int((~valid).sum())

cos_rows = []
for ds in ("voc", "coco"):
    rows, _, _, _ = draw_subset(ds)
    data = load_subset(ds, rows)
    for mode in METHODS:
        sims, n_zero = [], 0
        for full, objs in zip(data["full"], data[mode]):
            if len(objs) == 0:
                continue
            s, z = cosine_to_full(full, objs)
            sims.extend(s)
            n_zero += z
        sims = np.array(sims)
        cos_rows.append({
            "Dataset": ds.upper(),
            "Method": mode.capitalize(),
            "Images": int((data[f"{mode}_num_obj"] > 0).sum()),
            "Objects": len(sims),
            "ZeroVectors": n_zero,
            "CosSim_mean": sims.mean(),
            "CosSim_std": sims.std(),
            "CosSim_median": np.median(sims),
        })

cosine = pd.DataFrame(cos_rows).set_index(["Dataset", "Method"])
display(cosine.round(3))


In [ ]:
# ── Recall over all active concepts (no rank cutoff) ─────────────
# Share of all (image, label) pairs with at least one active synonym of the label (> 0)
rec_rows = []
for ds in ("voc", "coco"):
    rows, labels, _, _ = draw_subset(ds)
    data = load_subset(ds, rows)
    names, syn = CLASS_NAMES[ds], SYNONYMS[ds]
    label_sets = {col: [vocab_idx[w] for w in syn[n]] for col, n in enumerate(names) if n is not None}
    gt_per_img = [[c for c in np.flatnonzero(l) if label_sets[c]] for l in labels]
    for mode in METHODS:
        aggs = [aggregate(full, objs) for full, objs in zip(data["full"], data[mode])]
        for how in ("sum", "count"):
            hits = []
            for (s, cnt), gt in zip(aggs, gt_per_img):
                vec = s if how == "sum" else cnt
                hits.extend(bool((vec[label_sets[c]] > 0).any()) for c in gt)
            rec_rows.append({
                "Dataset": ds.upper(),
                "Method": mode.capitalize(),
                "Aggregation": how.capitalize(),
                "Pairs": len(hits),
                "Hits": int(np.sum(hits)),
                "Recall@all": np.mean(hits),
            })

recall_all = pd.DataFrame(rec_rows).set_index(["Dataset", "Method", "Aggregation"])
display(recall_all.round(3))
